# [SmoothQuant W8A8] EXAONE-4.0-1.2B 모델 양자화

SmoothQuant + INT8 W8A8 양자화 방식을 적용한 버전입니다.

**방식**: Weight 8-bit + Activation 8-bit (INT8×INT8 matmul)

**평가 서버 호환성**:
- vLLM 0.14.1 (compressed-tensors 네이티브 지원)
- compressed-tensors 0.13.0
- transformers 4.57.3
- AutoModelForCausalLM.from_pretrained() 호환

**vs GPTQ W4A16 베이스라인 비교**:
- 정확도: W8A8 > W4A16 (8-bit가 더 정밀)
- 속도: INT8×INT8 커널이 더 빠를 수 있음
- 파일 크기: W8A8 > W4A16 (약간 더 큼)

In [ ]:
!pip install llmcompressor

# Import

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

# Setting

In [ ]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# SmoothQuant + W8A8 Quantization
SCHEME = "W8A8"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens"]
SMOOTHING_STRENGTH = 0.8

# Model Loads

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

In [ ]:
import llmcompressor, pathlib, re

path = pathlib.Path(llmcompressor.__file__).resolve().parent / "modifiers" / "smoothquant" / "base.py"
txt = path.read_text().splitlines()

# def hook_fn(...) 라인 찾기
hook_idx = next((i for i, ln in enumerate(txt) if re.search(r"^\s*def\s+hook_fn\s*\(", ln)), None)
if hook_idx is None:
    raise RuntimeError("def hook_fn not found")

# hook_fn의 본문 들여쓰기 계산(다음 줄의 indent를 기준으로)
# 안전하게 다음 non-empty line을 찾음
body_idx = None
for i in range(hook_idx + 1, min(hook_idx + 50, len(txt))):
    if txt[i].strip():  # non-empty
        body_idx = i
        break
if body_idx is None:
    raise RuntimeError("hook_fn body not found")

indent = re.match(r"\s*", txt[body_idx]).group(0)

# 이미 early return이 들어갔는지 체크
marker = f"{indent}# [PATCH] disable smoothquant hook to avoid non-contiguous/Proxy tensor errors"
return_line = f"{indent}return"
if any(marker in ln for ln in txt[hook_idx:hook_idx+80]):
    print("[INFO] hook already patched")
else:
    # body_idx 위치에 2줄 삽입
    txt.insert(body_idx, return_line)
    txt.insert(body_idx, marker)

path.write_text("\n".join(txt) + "\n")
print("[OK] Disabled SmoothQuant hook in:", path)

# 리로드
import importlib
import llmcompressor.modifiers.smoothquant.base as sq_base
importlib.reload(sq_base)
print("[OK] Reloaded smoothquant.base")


In [ ]:
import importlib
import llmcompressor.modifiers.smoothquant.base as sq_base
importlib.reload(sq_base)
print("[OK] Reloaded smoothquant.base")


# SmoothQuant W8A8 Quantization

In [ ]:
import re
from llmcompressor.modifiers.smoothquant.utils import LayerMap
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

# (선택) 정규식 매칭 수 확인용
def count_matches(pattern: str) -> int:
    return sum(1 for n, _ in model.named_modules() if re.search(pattern, n))

print("[CHECK] q_proj:", count_matches(r"self_attn\.q_proj$"))
print("[CHECK] q_norm:", count_matches(r"self_attn\.q_norm$"))
print("[CHECK] k_proj:", count_matches(r"self_attn\.k_proj$"))
print("[CHECK] k_norm:", count_matches(r"self_attn\.k_norm$"))
print("[CHECK] post_ffn_ln:", count_matches(r"post_feedforward_layernorm$"))

mappings = [
    LayerMap(
        balance_layers=["re:.*self_attn\\.q_proj$"],
        smooth_layers="re:.*self_attn\\.q_norm$",
    ),
    LayerMap(
        balance_layers=["re:.*self_attn\\.k_proj$"],
        smooth_layers="re:.*self_attn\\.k_norm$",
    ),
    LayerMap(
        balance_layers=["re:.*gate_proj$", "re:.*up_proj$"],
        smooth_layers="re:.*post_feedforward_layernorm$",
    ),
]

print(f"[INFO] SmoothQuant W8A8 시작 (scheme={SCHEME}, smoothing={SMOOTHING_STRENGTH}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    QuantizationModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    ),
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] SmoothQuant W8A8 완료")


In [ ]:
import llmcompressor, pkgutil, inspect
import llmcompressor.modifiers.smoothquant as sq

print("llmcompressor version:", getattr(llmcompressor, "__version__", "unknown"))
print("smoothquant module:", sq.__file__)

# smoothquant 하위 모듈들 훑기
mods = [m.name for m in pkgutil.iter_modules(sq.__path__, sq.__name__ + ".")]
print("smoothquant submodules:\n", "\n".join(mods))

# LayerMap이 들어있는 모듈 찾기
hits = []
for m in mods:
    try:
        mod = __import__(m, fromlist=["*"])
        if hasattr(mod, "LayerMap"):
            hits.append((m, mod.LayerMap))
    except Exception:
        pass

print("LayerMap hits:", hits)


# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

# Submission

In [ ]:
zip_name = "submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")